## Example 1: Basic Valkey Client Operations

In [ ]:
import valkey
import json

# Example 1: Basic Valkey client setup and configuration
print("Basic Valkey client setup:")

# Create a Valkey client with various configuration options
clients = [
    valkey.Valkey(host='localhost', port=6379, decode_responses=True),
    valkey.Valkey(host='localhost', port=6379, db=1),
    valkey.Valkey(host='localhost', port=6379, socket_timeout=5.0),
    valkey.Valkey(host='localhost', port=6379, max_connections=20)
]

print(f"Created {len(clients)} Valkey client configurations:")
for i, client in enumerate(clients, 1):
    print(f"Client {i}: {type(client).__name__}")
    connection_kwargs = client.connection_pool.connection_kwargs
    print(f"  Host: {connection_kwargs.get('host', 'localhost')}")
    print(f"  Port: {connection_kwargs.get('port', 6379)}")
    print(f"  DB: {connection_kwargs.get('db', 0)}")

# Connection pool configuration
pool = valkey.ConnectionPool(
    host='localhost',
    port=6379,
    max_connections=10,
    decode_responses=True
)

pooled_client = valkey.Valkey(connection_pool=pool)
print(f"\nConnection pool client: {type(pooled_client).__name__}")
print(f"Max connections: {pool.max_connections}")

# Show available command methods
command_categories = {
    'string': ['get', 'set', 'append', 'incr', 'decr'],
    'list': ['lpush', 'rpush', 'lpop', 'rpop', 'llen'],
    'hash': ['hget', 'hset', 'hgetall', 'hdel', 'hkeys'],
    'set': ['sadd', 'srem', 'smembers', 'scard', 'sinter'],
    'sorted_set': ['zadd', 'zrem', 'zrange', 'zcard', 'zscore']
}

print(f"\nValkey command categories available:")
for category, commands in command_categories.items():
    available = [cmd for cmd in commands if hasattr(pooled_client, cmd)]
    print(f"  {category}: {len(available)}/{len(commands)} commands ({', '.join(available[:3])}...)")

print(f"Valkey client ready (Redis-compatible interface)")

## Example 2: Data Types and Operations

In [ ]:
import valkey
import json

# Example 2: Data Types and Operations
print("Valkey data types and operations:")

# Mock client for demonstration (since we don't have a running server)
class MockValkeyClient:
    def __init__(self):
        self.strings = {}
        self.lists = {}
        self.sets = {}
        self.hashes = {}
        self.sorted_sets = {}
    
    # String operations
    def set(self, key, value, ex=None):
        self.strings[key] = value
        return True
    
    def get(self, key):
        return self.strings.get(key)
    
    def incr(self, key):
        current = int(self.strings.get(key, 0))
        self.strings[key] = str(current + 1)
        return current + 1
    
    # List operations  
    def lpush(self, key, *values):
        if key not in self.lists:
            self.lists[key] = []
        for value in reversed(values):
            self.lists[key].insert(0, value)
        return len(self.lists[key])
    
    def lrange(self, key, start, end):
        if key not in self.lists:
            return []
        return self.lists[key][start:end+1 if end != -1 else None]
    
    # Hash operations
    def hset(self, name, key=None, value=None, mapping=None):
        if name not in self.hashes:
            self.hashes[name] = {}
        if mapping:
            self.hashes[name].update(mapping)
            return len(mapping)
        elif key and value:
            self.hashes[name][key] = value
            return 1
    
    def hgetall(self, name):
        return self.hashes.get(name, {})
    
    # Set operations
    def sadd(self, key, *values):
        if key not in self.sets:
            self.sets[key] = set()
        added = 0
        for value in values:
            if value not in self.sets[key]:
                self.sets[key].add(value)
                added += 1
        return added
    
    def smembers(self, key):
        return self.sets.get(key, set())

# Create mock client (would normally be valkey.Valkey(host='localhost', port=6379))
client = MockValkeyClient()

print("1. String operations:")
client.set("user:1:name", "Alice")
client.set("counter", "10")
print(f"   SET user:1:name -> GET: {client.get('user:1:name')}")
print(f"   INCR counter: {client.incr('counter')}")
print(f"   INCR counter: {client.incr('counter')}")

print("\n2. List operations:")
client.lpush("tasks", "task1", "task2", "task3")
tasks = client.lrange("tasks", 0, -1)
print(f"   LPUSH tasks -> LRANGE: {tasks}")

print("\n3. Hash operations:")
user_profile = {
    "name": "Bob",
    "email": "bob@example.com", 
    "age": "25"
}
client.hset("user:2:profile", mapping=user_profile)
profile = client.hgetall("user:2:profile")
print(f"   HSET user profile -> HGETALL: {profile}")

print("\n4. Set operations:")
client.sadd("tags", "python", "redis", "database", "python")  # duplicate python
members = client.smembers("tags")
print(f"   SADD tags -> SMEMBERS: {members}")

# Show actual valkey client initialization
print(f"\n5. Real Valkey client configuration:")
try:
    real_client = valkey.Valkey(
        host='localhost',
        port=6379, 
        decode_responses=True,
        socket_timeout=5,
        max_connections=10
    )
    print(f"   Client type: {type(real_client).__name__}")
    print(f"   Connection pool: {type(real_client.connection_pool).__name__}")
except Exception as e:
    print(f"   Note: Real server needed for actual operations")
    raise

print("Valkey supports all Redis data types and operations")

## Example 3: Connection Management and Configuration

In [ ]:
import valkey
import time

# Example 3: Connection Management and Configuration
print("Connection management and configuration:")

# Different connection configurations
configurations = [
    {
        "name": "Basic Connection",
        "config": {"host": "localhost", "port": 6379}
    },
    {
        "name": "With Timeouts", 
        "config": {
            "host": "localhost",
            "port": 6379,
            "socket_timeout": 5.0,
            "socket_connect_timeout": 2.0
        }
    },
    {
        "name": "With Database Selection",
        "config": {"host": "localhost", "port": 6379, "db": 1}
    },
    {
        "name": "With Decoding",
        "config": {
            "host": "localhost", 
            "port": 6379,
            "decode_responses": True,
            "encoding": "utf-8"
        }
    }
]

print("1. Client configurations:")
clients = []
for i, conf in enumerate(configurations, 1):
    try:
        client = valkey.Valkey(**conf["config"])
        clients.append(client)
        print(f"   {i}. {conf['name']}: ✓ Created")
        
        # Show connection details
        pool = client.connection_pool
        print(f"      Host: {pool.connection_kwargs.get('host')}")
        print(f"      Port: {pool.connection_kwargs.get('port')}")
        print(f"      DB: {pool.connection_kwargs.get('db', 0)}")
        print(f"      Decode responses: {pool.connection_kwargs.get('decode_responses', False)}")
        
    except Exception as e:
        print(f"   {i}. {conf['name']}: ✗ Error - {e}")
        raise

# Connection pooling
print(f"\n2. Connection pooling:")
pool_config = {
    "host": "localhost",
    "port": 6379,
    "max_connections": 20,
    "retry_on_timeout": True,
    "health_check_interval": 30
}

try:
    connection_pool = valkey.ConnectionPool(**pool_config)
    pooled_client = valkey.Valkey(connection_pool=connection_pool)
    
    print(f"   Pool created with max {connection_pool.max_connections} connections")
    print(f"   Retry on timeout: {pool_config['retry_on_timeout']}")
    print(f"   Health check interval: {pool_config['health_check_interval']}s")
    
except Exception as e:
    print(f"   Pool creation error: {e}")
    raise

# URL-based connection
print(f"\n3. URL-based connections:")
url_examples = [
    "redis://localhost:6379/0",
    "redis://localhost:6379/1?socket_timeout=5",
    "rediss://localhost:6380/0",  # SSL
    "unix:///tmp/redis.sock?db=0"  # Unix socket
]

for url in url_examples:
    try:
        # Would normally create client from URL
        print(f"   URL: {url}")
        print(f"       Format: {'✓ Valid' if url.startswith(('redis://', 'rediss://', 'unix://')) else '✗ Invalid'}")
        
    except Exception as e:
        print(f"   URL {url}: Error - {e}")
        raise

# Health checking
print(f"\n4. Health and monitoring:")

class MockHealthChecker:
    def __init__(self, client_name):
        self.client_name = client_name
        
    def ping(self):
        # Simulate ping response
        return True
        
    def info(self, section=None):
        # Simulate info command
        return {
            "valkey_version": "7.2.0",
            "connected_clients": "2",
            "used_memory": "1048576",
            "total_connections_received": "150"
        }

# Mock health check
checker = MockHealthChecker("test_client")
try:
    ping_result = checker.ping()
    info_result = checker.info()
    
    print(f"   Ping test: {'✓ OK' if ping_result else '✗ FAIL'}")
    print(f"   Server info:")
    for key, value in info_result.items():
        print(f"     {key}: {value}")
        
except Exception as e:
    print(f"   Health check error: {e}")
    raise

# Best practices
print(f"\n5. Best practices:")
best_practices = [
    "Use connection pooling for production applications",
    "Set appropriate socket timeouts to avoid hanging",
    "Enable retry_on_timeout for resilience",
    "Use decode_responses=True for string operations", 
    "Monitor connection health with ping/info commands",
    "Configure max_connections based on your workload"
]

for i, practice in enumerate(best_practices, 1):
    print(f"   {i}. {practice}")

print("\nValkey provides flexible connection management for various deployment scenarios")

## Example 4: Pipeline Operations and Transactions

In [ ]:
import valkey
import time

# Example 4: Pipeline Operations and Transactions
print("Pipeline operations and transactions:")

# Mock pipeline for demonstration
class MockPipeline:
    def __init__(self):
        self.commands = []
        self.results = []
    
    def set(self, key, value):
        self.commands.append(("SET", key, value))
        return self
    
    def get(self, key):
        self.commands.append(("GET", key))
        return self
    
    def incr(self, key):
        self.commands.append(("INCR", key))
        return self
    
    def lpush(self, key, *values):
        self.commands.append(("LPUSH", key, values))
        return self
    
    def execute(self):
        # Simulate executing all commands atomically
        self.results = []
        for cmd in self.commands:
            if cmd[0] == "SET":
                self.results.append(True)
            elif cmd[0] == "GET":
                self.results.append(f"value_for_{cmd[1]}")
            elif cmd[0] == "INCR":
                self.results.append(42)  # Mock increment result
            elif cmd[0] == "LPUSH":
                self.results.append(len(cmd[2]))
        return self.results
    
    def reset(self):
        self.commands = []
        self.results = []

# Create mock client with pipeline support
class MockValkeyWithPipeline:
    def __init__(self):
        self.data = {}
    
    def pipeline(self, transaction=False):
        return MockPipeline()

client = MockValkeyWithPipeline()

print("1. Basic pipeline operations:")
pipe = client.pipeline()

# Queue up multiple commands
pipe.set("user:1", "Alice")
pipe.set("user:2", "Bob") 
pipe.set("counter", "0")
pipe.incr("counter")
pipe.incr("counter")
pipe.get("user:1")
pipe.get("counter")

print(f"   Queued {len(pipe.commands)} commands")
for i, cmd in enumerate(pipe.commands, 1):
    print(f"     {i}. {cmd[0]} {' '.join(str(x) for x in cmd[1:])}")

# Execute pipeline
start_time = time.time()
results = pipe.execute()
execution_time = time.time() - start_time

print(f"\n   Executed in {execution_time:.4f}s")
print(f"   Results: {results}")

print("\n2. Performance comparison - Pipeline vs Individual:")

# Simulate individual commands timing
individual_start = time.time()
for cmd in pipe.commands:
    time.sleep(0.001)  # Simulate network latency per command
individual_time = time.time() - individual_start

# Simulate pipeline timing (single round trip)
pipeline_start = time.time()
time.sleep(0.001)  # Single network round trip
pipeline_time = time.time() - pipeline_start

print(f"   Individual commands: {individual_time:.4f}s ({len(pipe.commands)} round trips)")
print(f"   Pipeline commands: {pipeline_time:.4f}s (1 round trip)")
print(f"   Performance improvement: {individual_time/pipeline_time:.1f}x faster")

print("\n3. Transaction pipeline (MULTI/EXEC):")

# Mock transaction pipeline
class MockTransactionPipeline(MockPipeline):
    def __init__(self):
        super().__init__()
        self.transaction = True
        
    def multi(self):
        self.commands.append(("MULTI",))
        return self
        
    def exec(self):
        self.commands.append(("EXEC",))
        return self.execute()

tx_pipe = MockTransactionPipeline()
tx_pipe.multi()
tx_pipe.set("account:1:balance", "1000")
tx_pipe.set("account:2:balance", "500")
tx_pipe.incr("transaction:count")
results = tx_pipe.exec()

print(f"   Transaction with {len(tx_pipe.commands)-2} operations")  # -2 for MULTI/EXEC
print(f"   All operations executed atomically")
print(f"   Results: {results}")

print("\n4. Pipeline patterns:")

patterns = [
    {
        "name": "Batch Operations",
        "description": "Group related operations for efficiency",
        "commands": ["SET user:1 data", "SET user:2 data", "SET user:3 data"]
    },
    {
        "name": "Read-Heavy Workload", 
        "description": "Batch multiple GET operations",
        "commands": ["GET user:1", "GET user:2", "GET user:3", "GET config:app"]
    },
    {
        "name": "Counter Updates",
        "description": "Atomic counter operations", 
        "commands": ["INCR page:views", "INCR user:logins", "INCR api:calls"]
    },
    {
        "name": "Data Migration",
        "description": "Move data between keys/databases",
        "commands": ["GET old:key", "SET new:key value", "DEL old:key"]
    }
]

for pattern in patterns:
    print(f"   {pattern['name']}:")
    print(f"     Description: {pattern['description']}")
    print(f"     Commands: {len(pattern['commands'])} operations")

print("\n5. Pipeline best practices:")

best_practices = [
    "Batch related operations to reduce network round trips",
    "Use transactions (MULTI/EXEC) for atomic operations",
    "Limit pipeline size to avoid memory issues (< 1000 commands)",
    "Handle pipeline errors gracefully with try/except",
    "Consider using pipeline for read-heavy workloads",
    "Monitor pipeline performance with timing metrics"
]

for i, practice in enumerate(best_practices, 1):
    print(f"   {i}. {practice}")

# Show actual valkey pipeline usage
print(f"\nActual Valkey pipeline usage:")
try:
    real_client = valkey.Valkey(host='localhost', port=6379)
    print(f"   # Create pipeline")
    print(f"   pipe = client.pipeline()")
    print(f"   # Queue commands") 
    print(f"   pipe.set('key1', 'value1')")
    print(f"   pipe.get('key1')")
    print(f"   # Execute atomically")
    print(f"   results = pipe.execute()")
    
except Exception as e:
    print(f"   Note: Requires running Valkey server")
    raise

print("\nPipelines provide significant performance improvements for batch operations")

## Example 5: Advanced Features and Production Patterns

In [ ]:
import valkey
import json
import time
import hashlib

# Example 5: Advanced Features and Production Patterns
print("Advanced features and production patterns:")

# Mock advanced client
class AdvancedMockClient:
    def __init__(self):
        self.data = {}
        self.expires = {}
        self.pub_sub_channels = {}
    
    def setex(self, key, time_seconds, value):
        self.data[key] = value
        self.expires[key] = time.time() + time_seconds
        return True
    
    def ttl(self, key):
        if key in self.expires:
            remaining = self.expires[key] - time.time()
            return max(0, int(remaining))
        return -1
    
    def exists(self, *keys):
        return sum(1 for key in keys if key in self.data)
    
    def scan_iter(self, match=None):
        # Simulate scanning keys
        for key in self.data.keys():
            if match is None or match.replace('*', '') in key:
                yield key
    
    def mget(self, *keys):
        return [self.data.get(key) for key in keys]
    
    def mset(self, mapping):
        self.data.update(mapping)
        return True

client = AdvancedMockClient()

print("1. TTL and expiration management:")
# Set keys with expiration
client.setex("session:abc123", 3600, "session_data")
client.setex("cache:temp", 60, "temporary_data") 
client.setex("token:xyz", 1800, "auth_token")

# Check TTL
keys = ["session:abc123", "cache:temp", "token:xyz"]
for key in keys:
    ttl = client.ttl(key)
    status = "expires" if ttl > 0 else "permanent"
    print(f"   {key}: {ttl}s remaining ({status})")

print(f"   Keys exist: {client.exists(*keys)}/{len(keys)}")

print("\n2. Batch operations with MGET/MSET:")
# Set multiple keys at once
user_data = {
    "user:1:name": "Alice",
    "user:1:email": "alice@example.com",
    "user:1:role": "admin"
}
client.mset(user_data)
print(f"   Set {len(user_data)} keys with MSET")

# Get multiple keys at once
user_keys = list(user_data.keys())
values = client.mget(*user_keys)
print(f"   Retrieved {len(values)} values with MGET:")
for key, value in zip(user_keys, values):
    print(f"     {key}: {value}")

print("\n3. Key scanning and pattern matching:")
# Add some test keys
test_keys = {
    "user:1:profile": "profile1",
    "user:2:profile": "profile2", 
    "session:abc": "sess1",
    "session:def": "sess2",
    "cache:page:1": "page1",
    "cache:page:2": "page2"
}
client.data.update(test_keys)

# Scan for different patterns
patterns = [
    ("user:*", "user keys"),
    ("session:*", "session keys"),
    ("cache:*", "cache keys")
]

for pattern, description in patterns:
    matching_keys = list(client.scan_iter(match=pattern))
    print(f"   {description}: {len(matching_keys)} keys")
    for key in matching_keys[:3]:  # Show first 3
        print(f"     - {key}")

print("\n4. Caching patterns:")

class ValkeyCache:
    def __init__(self, client):
        self.client = client
        self.stats = {"hits": 0, "misses": 0}
    
    def get_or_set(self, key, compute_func, ttl=300):
        """Cache-aside pattern with statistics"""
        value = self.client.data.get(key)
        if value is not None:
            self.stats["hits"] += 1
            try:
                return json.loads(value)
            except (json.JSONDecodeError, TypeError):
                return value
        
        # Cache miss - compute and store
        self.stats["misses"] += 1
        computed_value = compute_func()
        
        try:
            serialized = json.dumps(computed_value)
        except TypeError:
            serialized = str(computed_value)
        
        self.client.setex(key, ttl, serialized)
        return computed_value
    
    def invalidate(self, pattern):
        """Invalidate cached entries by pattern"""
        count = 0
        for key in list(self.client.scan_iter(match=pattern)):
            if key in self.client.data:
                del self.client.data[key]
                count += 1
        return count

cache = ValkeyCache(client)

# Test caching
def expensive_computation(n):
    time.sleep(0.01)  # Simulate work
    return {"result": n * n, "computed_at": time.time()}

# First call - cache miss
result1 = cache.get_or_set("compute:5", lambda: expensive_computation(5), ttl=120)
print(f"   First call result: {result1}")

# Second call - cache hit
result2 = cache.get_or_set("compute:5", lambda: expensive_computation(5), ttl=120)
print(f"   Second call result: {result2}")

print(f"   Cache stats: {cache.stats}")
print(f"   Cache hit ratio: {cache.stats['hits']/(cache.stats['hits']+cache.stats['misses']):.1%}")

print("\n5. Production monitoring and health checks:")

class ValkeyMonitor:
    def __init__(self, client):
        self.client = client
    
    def health_check(self):
        """Comprehensive health check"""
        try:
            # Simulate various health checks
            checks = {
                "connection": True,  # client.ping()
                "memory_usage": "1.2MB",  # Mock memory info
                "connected_clients": 5,
                "total_commands_processed": 12345,
                "keyspace_hits": 8900,
                "keyspace_misses": 1100
            }
            
            # Calculate hit ratio
            hits = checks["keyspace_hits"]
            misses = checks["keyspace_misses"]
            hit_ratio = hits / (hits + misses) if (hits + misses) > 0 else 0
            
            return {
                "status": "healthy",
                "checks": checks,
                "metrics": {
                    "hit_ratio": f"{hit_ratio:.1%}",
                    "total_ops": hits + misses
                }
            }
        except Exception as e:
            return {"status": "unhealthy", "error": str(e)}
    
    def performance_metrics(self):
        """Performance metrics for monitoring"""
        return {
            "response_time_ms": 0.5,  # Mock latency
            "throughput_ops_sec": 10000,
            "connection_pool_usage": 0.6,
            "memory_fragmentation_ratio": 1.15
        }

monitor = ValkeyMonitor(client)
health = monitor.health_check()
metrics = monitor.performance_metrics()

print(f"   Health status: {health['status']}")
print(f"   Connection: {'✓' if health['checks']['connection'] else '✗'}")
print(f"   Memory usage: {health['checks']['memory_usage']}")
print(f"   Hit ratio: {health['metrics']['hit_ratio']}")

print(f"\n   Performance metrics:")
for metric, value in metrics.items():
    print(f"     {metric}: {value}")

# Best practices
print(f"\n6. Production best practices:")
practices = [
    "Implement proper connection pooling with health checks",
    "Use key expiration to prevent memory leaks", 
    "Monitor hit ratios and adjust caching strategies",
    "Use pipelines for batch operations to reduce latency",
    "Implement graceful fallbacks when Valkey is unavailable",
    "Set up monitoring and alerting for key metrics",
    "Use consistent key naming patterns for easier management",
    "Regularly scan and clean up unused keys"
]

for i, practice in enumerate(practices, 1):
    print(f"   {i}. {practice}")

print(f"\nValkey provides enterprise-grade features for production Redis workloads")